# 05 — Summarization & Entity Recognition
### DiscoverAI · Deloitte × LUISS

Covers **Deloitte Objective 2**: Review-Driven Product Insights & Summarization.

**What this notebook produces:**
- `product_summaries.csv` — structured summaries (full text, pros, cons, best_for)
- `product_entities.csv` — extracted entities (ingredients, certifications, use cases)
- `product_profiles.csv` — merged file combining both with the embedding index

**Approach:**
- **Summarization**: extractive + abstractive pipeline using `facebook/bart-large-cnn`
  - For products with ≥5 reviews: BART generates a fluent abstractive summary
  - For products with <5 reviews: extractive fallback (most informative sentences)
- **Entity Recognition**: custom regex patterns + spaCy NER
  - Ingredients: vitamin C, hyaluronic acid, zinc oxide, biotin, etc.
  - Certifications: organic, cruelty-free, paraben-free, fragrance-free, etc.
  - Use cases: for dry skin, for joint pain, for babies, etc.

**⚠️ Runtime note:**
On M1 Mac CPU, BART takes ~2s/product → ~6h for 11k products.
**Run on Google Colab with T4 GPU** → ~67 minutes.
Checkpoints are saved every 500 products — safe to interrupt and resume.


## 0 · Setup and dependencies

In [ ]:
import subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

# Install missing dependencies
for pkg in ["gradio", "spacy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# Download spaCy model if not present
try:
    import spacy
    spacy.load("en_core_web_sm")
    print("spaCy en_core_web_sm: already installed")
except OSError:
    print("Downloading spaCy en_core_web_sm...")
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)

print("Dependencies ready.")

In [ ]:
import os, re, warnings
import time
import numpy as np
import pandas as pd
from collections import defaultdict
import spacy
from transformers import pipeline

warnings.filterwarnings("ignore")

DATA_DIR = "/Users/danielegiovanardi/MSTERRORS/Mean-Squared-Terrors/data"

products = pd.read_csv(os.path.join(DATA_DIR, "products_cleaned.csv"))
reviews  = pd.read_csv(os.path.join(DATA_DIR, "reviews_cleaned.csv"))

import torch
DEVICE = 0 if torch.cuda.is_available() else -1  # -1 = CPU (MPS non supportato da transformers pipeline)

print(f"Prodotti: {len(products):,}  |  Review: {len(reviews):,}")
print(f"Device per transformers: {'GPU' if DEVICE == 0 else 'CPU'}")


## 1 · Entity recognition

Extracts three types of structured information from product text and reviews:

1. **Ingredients**: common active ingredients (regex patterns)
2. **Certifications**: quality/safety claims (organic, cruelty-free, etc.)
3. **Use cases**: target audience/use (for dry skin, for joint pain, etc.)
4. **Brand entities**: organisations mentioned (spaCy NER)

Low overall coverage (9-20%) reflects that most products lack structured
ingredient lists in their Amazon metadata.


In [ ]:
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

# Custom regex patterns for ingredients and certifications
INGREDIENT_PATTERN = (
    r'\b(vitamin [a-z]\d{0,2}|hyaluronic acid|retinol|niacinamide|salicylic acid|'
    r'benzoyl peroxide|glycolic acid|collagen|keratin|biotin|caffeine|'
    r'aloe vera|tea tree|coconut oil|argan oil|jojoba|shea butter|'
    r'zinc oxide|titanium dioxide|spf\s*\d+|melatonin|magnesium|'
    r'probiotics?|prebiotics?|omega[- ]?3|glucosamine|turmeric|'
    r'activated charcoal|witch hazel|apple cider vinegar)\b'
)

CERT_PATTERN = (
    r'\b(organic|natural|vegan|cruelty.?free|paraben.?free|sulfate.?free|'
    r'fragrance.?free|hypoallergenic|dermatologist tested|fda|clinically tested|'
    r'non.?gmo|gluten.?free|alcohol.?free|oil.?free|non.?comedogenic)\b'
)

USE_PATTERNS = [
    r'\bfor (dry|oily|sensitive|combination|acne.?prone|mature|aging|normal) skin\b',
    r'\bfor (dry|damaged|fine|thinning|color.?treated|curly|straight) hair\b',
    r'\bfor (back|joint|muscle|knee|neck|shoulder) pain\b',
    r'\bfor (babies?|infants?|kids?|children|toddlers?)\b',
    r'\bfor (men|women|adults?|seniors?)\b',
]

def extract_entities(text, asin=""):
    """Extract ingredients, certifications, use cases and brand entities from text."""
    if not text or pd.isna(text):
        return {"ingredients": [], "certifications": [], "use_cases": [], "brands": []}

    text_lower = str(text).lower()

    # Ingredient patterns
    matches = re.findall(INGREDIENT_PATTERN, text_lower, re.IGNORECASE)
    ingredients = list(set(m.lower().strip() for m in matches))

    # Certification patterns
    matches = re.findall(CERT_PATTERN, text_lower, re.IGNORECASE)
    certs = list(set(m.lower().strip() for m in matches))

    # Use case patterns
    use_cases = []
    for pattern in USE_PATTERNS:
        matches = re.findall(pattern, text_lower, re.IGNORECASE)
        use_cases.extend([m.lower().strip() if isinstance(m, str) else " ".join(m).lower().strip()
                         for m in matches])
    use_cases = list(set(use_cases))

    # Named entities with spaCy (product text only — reviews too slow)
    brands = []
    if len(text) < 1000:  # solo testi corti per velocità
        doc = nlp(text[:500])
        brands = list(set([ent.text.strip() for ent in doc.ents
                          if ent.label_ in ("ORG", "PRODUCT", "BRAND")
                          and len(ent.text) > 2]))

    return {
        "ingredients":    ingredients,
        "certifications": certs,
        "use_cases":      use_cases,
        "brands":         brands[:5],  # max 5
    }

# Quick test
sample = products.iloc[5]["product_text_base"]
print(f"Test on: '{sample[:80]}...'")
result = extract_entities(sample)
for k, v in result.items():
    print(f"  {k}: {v}")


In [ ]:
# Apply to all products
print(f"Extracting entities from {len(products):,} products...")
entity_records = []

for i, (_, row) in enumerate(products.iterrows()):
    ents = extract_entities(row["product_text_base"], row["parent_asin"])

    # Also extract from top-3 reviews (ingredients and certifications only)
    rev_text = " ".join(
        reviews[reviews["parent_asin"] == row["parent_asin"]]["text"]
        .fillna("").tolist()[:3]  # prime 3 review per velocità
    )
    rev_ents = extract_entities(rev_text[:1000])

    # Merge and deduplicate
    all_ings  = list(set(ents["ingredients"] + rev_ents["ingredients"]))
    all_certs = list(set(ents["certifications"] + rev_ents["certifications"]))
    all_uses  = list(set(ents["use_cases"] + rev_ents["use_cases"]))

    entity_records.append({
        "parent_asin":    row["parent_asin"],
        "product_title":  row["product_title"],
        "ingredients":    " | ".join(all_ings) if all_ings else "",
        "certifications": " | ".join(all_certs) if all_certs else "",
        "use_cases":      " | ".join(all_uses) if all_uses else "",
        "brands_mentioned": " | ".join(ents["brands"]) if ents["brands"] else "",
        "n_ingredients":  len(all_ings),
        "n_certifications": len(all_certs),
    })

    if (i+1) % 1000 == 0:
        print(f"  {i+1:,}/{len(products):,}")

entities_df = pd.DataFrame(entity_records)
print(f"\nEntities extracted for {len(entities_df):,} products")
print(f"With at least 1 ingredient:     {(entities_df['n_ingredients']>0).mean():.1%}")
print(f"With at least 1 certification:  {(entities_df['n_certifications']>0).mean():.1%}")
print(f"With at least 1 use case:        {(entities_df['use_cases']!='').mean():.1%}")

entities_df.to_csv(os.path.join(DATA_DIR, "product_entities.csv"), index=False)
print(f"\nSaved product_entities.csv")

# Mostra esempi
print("\n--- 3 examples ---")
for _, r in entities_df[entities_df['n_ingredients']>2].head(3).iterrows():
    print(f"\n{r['product_title'][:60]}")
    print(f"  Ingredienti: {r['ingredients'][:100]}")
    print(f"  Cert: {r['certifications'][:80]}")
    print(f"  Use: {r['use_cases'][:80]}")


## 2 · Review summarization

**Approach: extractive + abstractive**

1. Select the most informative sentences from reviews by bucket:
   - Positive reviews (≥4 stars) → `pros`
   - Negative reviews (≤2 stars) → `cons`
   - All reviews → `best_for` context
2. Combine into input text for BART
3. BART generates a fluent abstractive summary

Products with <5 reviews use extractive-only (no BART call).

**⚠️ On Colab T4 this takes ~67 minutes. Checkpoints every 500 products.**


In [ ]:
# Carica modello di summarization
print("Loading BART summarization model...")
print("(first run: downloads ~1.6GB — cached afterwards)")
import time
t0 = time.time()

from transformers import BartForConditionalGeneration, BartTokenizerFast
import torch

_model_name = "facebook/bart-large-cnn"
_bart_tokenizer = BartTokenizerFast.from_pretrained(_model_name)
_bart_model = BartForConditionalGeneration.from_pretrained(_model_name)
if DEVICE == 0:
    _bart_model = _bart_model.cuda()

# Wrapper con la stessa interfaccia del pipeline
def summarizer(text, max_length=80, min_length=20, do_sample=False, **kwargs):
    inputs = _bart_tokenizer(
        text, return_tensors="pt", max_length=1024, truncation=True
    )
    if DEVICE == 0:
        inputs = {k: v.cuda() for k, v in inputs.items()}
    summary_ids = _bart_model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        do_sample=do_sample,
        num_beams=4,
        early_stopping=True,
    )
    out = _bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return [{"summary_text": out}]

print(f"Model loaded in {time.time()-t0:.1f}s")


In [ ]:
def extract_key_sentences(texts, max_sents=5):
    """
    Extract the most informative sentences from a list of texts.
    Criteria: length > 12 words, does not start with personal 'I', under 300 chars.
    """
    all_sents = []
    for text in texts:
        if not text or pd.isna(text):
            continue
        sents = re.split(r'(?<=[.!?])\s+', str(text))
        for s in sents:
            s = s.strip()
            words = s.split()
            if (len(words) >= 12 and               # abbastanza lunga
                not s.lower().startswith("i ") and  # non personale
                len(s) < 300):                      # non troppo lunga
                all_sents.append(s)

    # Approximate deduplication: remove near-duplicate sentences
    unique = []
    for s in all_sents:
        if not any(s[:40] in u for u in unique):
            unique.append(s)

    return unique[:max_sents]


def summarize_product(asin, prod_row, rev_df, summarizer, use_model=True):
    """
    Generate a structured summary for one product.

    Returns dict with:
      - summary_full: fluent narrative summary (BART or extractive)
      - pros: key positive points from ≥4-star reviews
      - cons: key issues from ≤2-star reviews
      - best_for: identified use case
      - method: 'model' (BART) or 'extractive' (fallback)
    """
    prod_revs = rev_df[rev_df["parent_asin"] == asin]

    pos_revs = prod_revs[prod_revs["rating"] >= 4]["text"].dropna().tolist()
    neg_revs = prod_revs[prod_revs["rating"] <= 2]["text"].dropna().tolist()
    all_revs = prod_revs["text"].dropna().tolist()

    # Extract key sentences per sentiment bucket
    pos_sents = extract_key_sentences(pos_revs, max_sents=4)
    neg_sents = extract_key_sentences(neg_revs, max_sents=3)
    all_sents = extract_key_sentences(all_revs, max_sents=5)

    # Build BART input text
    input_parts = []
    title = str(prod_row.get("product_title", ""))[:100]
    input_parts.append(f"Product: {title}.")

    if pos_sents:
        input_parts.append("Positive reviews: " + " ".join(pos_sents[:2]))
    if neg_sents:
        input_parts.append("Negative feedback: " + " ".join(neg_sents[:1]))
    if not pos_sents and not neg_sents:
        # Fallback su product_text_base
        input_parts.append(str(prod_row.get("product_text_base", ""))[:400])

    input_text = " ".join(input_parts)[:800]  # max 800 char per BART

    # Generate summary with BART (if text is long enough)
    summary_full = ""
    method = "extractive"

    if use_model and len(input_text) > 100:
        try:
            result = summarizer(
                input_text,
                max_length=80,
                min_length=20,
                do_sample=False,
            )
            summary_full = result[0]["summary_text"]
            method = "model"
        except Exception:
            summary_full = ""

    # Extractive fallback
    if not summary_full and all_sents:
        summary_full = all_sents[0][:200]

    # Pros and cons as plain text
    pros = pos_sents[0][:150] if pos_sents else ""
    cons = neg_sents[0][:150] if neg_sents else ""

    # Best for: find sentence mentioning skin/hair/pain type
    best_for = ""
    if all_sents:
        # Cerca frasi con "for" + tipo di pelle/capelli/dolore
        for s in all_sents:
            if re.search(r'\bfor (dry|oily|sensitive|damaged|fine|mature)', s, re.I):
                best_for = s[:100]
                break

    return {
        "parent_asin":  asin,
        "summary_full": summary_full,
        "pros":         pros,
        "cons":         cons,
        "best_for":     best_for,
        "method":       method,
        "n_reviews_used": len(prod_revs),
    }

# Test rapido su 3 prodotti
print("Testing summarization on 3 products...")
test_asins = products[products["product_text_base"].str.len() > 100]["parent_asin"].head(3).tolist()
for asin in test_asins:
    prod_row = products[products["parent_asin"] == asin].iloc[0]
    result   = summarize_product(asin, prod_row, reviews, summarizer, use_model=True)
    print(f"\n--- {str(prod_row['product_title'])[:60]}... ---")
    print(f"  Method : {result['method']}")
    print(f"  Summary: {result['summary_full'][:200]}")
    print(f"  Pros   : {result['pros'][:100]}")
    print(f"  Cons   : {result['cons'][:100]}")


In [ ]:
# Batch summarization — all products ───────────────────────────────────
# Strategy: BART for products with >=5 reviews, extractive for the rest
# Questo riduce il tempo da 8h a ~2h su M1

# RECOMMENDED: run on Google Colab with T4 GPU (~67 min vs ~6h on M1)

print("Starting batch summarization...")
print("Estimated time: ~2s/product with BART on M1 CPU")
print("  Products with >=5 reviews (BART model): ~40 min on GPU")
print("  Products with <5 reviews (extractive): fast")
print("Press Interrupt to stop — partial results are saved every 500 products.\n")

summaries = []
SAVE_EVERY = 500   # salva checkpoint ogni 500 prodotti
t0 = time.time()

for i, (_, prod_row) in enumerate(products.iterrows()):
    asin      = prod_row["parent_asin"]
    n_reviews = len(reviews[reviews["parent_asin"] == asin])

    # Use BART only for products with enough reviews
    use_model = n_reviews >= 5

    result = summarize_product(asin, prod_row, reviews, summarizer, use_model=use_model)
    summaries.append(result)

    # Progress log every 100 products
    if (i + 1) % 100 == 0:
        elapsed  = time.time() - t0
        per_prod = elapsed / (i + 1)
        remaining = per_prod * (len(products) - i - 1)
        print(f"  {i+1:,}/{len(products):,}  |  {elapsed/60:.1f} min elapsed  "
              f"|  ~{remaining/60:.0f} min remaining")

    # Checkpoint save
    if (i + 1) % SAVE_EVERY == 0:
        df_partial = pd.DataFrame(summaries)
        df_partial.to_csv(os.path.join(DATA_DIR, "product_summaries_partial.csv"), index=False)
        print(f"  [checkpoint salvato: {len(summaries)} prodotti]")

# Final save
summaries_df = pd.DataFrame(summaries)
summaries_df.to_csv(os.path.join(DATA_DIR, "product_summaries.csv"), index=False)

elapsed = time.time() - t0
print(f"\nCompleted in {elapsed/60:.1f} min")
print(f"Products with BART summary:    {(summaries_df['method']=='model').sum():,}")
print(f"Products with extractive:      {(summaries_df['method']=='extractive').sum():,}")
print(f"Products without summary:      {(summaries_df['summary_full']=='').sum():,}")
print(f"\nSaved product_summaries.csv")


## 3 · Quality check

In [ ]:
# Reload if notebook was interrupted
if 'summaries_df' not in dir() or len(summaries_df) == 0:
    summaries_df = pd.read_csv(os.path.join(DATA_DIR, "product_summaries.csv"))
    print(f"Reloaded: {len(summaries_df):,} summaries")

# Statistiche
print(f"Total summaries: {len(summaries_df):,}")
print(f"With non-empty summary: {(summaries_df['summary_full']!='').mean():.1%}")
print(f"With pros: {(summaries_df['pros']!='').mean():.1%}")
print(f"With cons: {(summaries_df['cons']!='').mean():.1%}")
print(f"BART method: {(summaries_df['method']=='model').mean():.1%}")

# Mostra 5 esempi di alta qualità
print("\n--- Sample summaries (BART method) ---")
model_summs = summaries_df[
    (summaries_df['method'] == 'model') &
    (summaries_df['pros'] != '') &
    (summaries_df['cons'] != '')
].head(5)

for _, r in model_summs.iterrows():
    prod_title = products[products['parent_asin'] == r['parent_asin']]['product_title'].values
    title = prod_title[0][:60] if len(prod_title) > 0 else r['parent_asin']
    print(f"\n{title}")
    print(f"  Summary: {r['summary_full']}")
    print(f"  Pros:    {r['pros'][:120]}")
    print(f"  Cons:    {r['cons'][:120]}")
    if r['best_for']:
        print(f"  Best for: {r['best_for'][:100]}")


## 4 · Merge with embedding index

Combines summaries and entities into a single `product_profiles.csv`.
This is the main file used by notebooks 04 and 06.


In [ ]:
# Load embedding_index_enriched and merge with summaries and entities
idx = pd.read_csv(os.path.join(DATA_DIR, "embedding_index_enriched.csv"))

# Merge summaries
idx = idx.merge(
    summaries_df[["parent_asin","summary_full","pros","cons","best_for","method"]],
    on="parent_asin", how="left"
)

# Merge entities
ents = pd.read_csv(os.path.join(DATA_DIR, "product_entities.csv"))
idx = idx.merge(
    ents[["parent_asin","ingredients","certifications","use_cases","n_ingredients","n_certifications"]],
    on="parent_asin", how="left"
)

idx.to_csv(os.path.join(DATA_DIR, "product_profiles.csv"), index=False)
print(f"product_profiles.csv saved: {len(idx):,} rows")
print(f"Columns: {list(idx.columns)}")
print(f"\nCoverage:")
print(f"  With summary:      {idx['summary_full'].notna().mean():.1%}")
print(f"  With pros:         {(idx['pros'].fillna('')!='').mean():.1%}")
print(f"  With ingredients:  {(idx['ingredients'].fillna('')!='').mean():.1%}")
print(f"  With certifications: {(idx['certifications'].fillna('')!='').mean():.1%}")
